In [1]:
import os
import cv2
import numpy as np
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile

In [2]:
# Define the paths to your image folders
healthy_path = '../dataset_resized_512x512/healthy'
caterpillar_path = '../dataset_resized_512x512/caterpillar'
leaf_spot_path = '../dataset_resized_512x512/leaf_spot'

In [3]:
# Define the output directories for train, test, and validation sets
output_dir = '../dataset_resized_512x512/split'
train_dir = os.path.join(output_dir, 'train')
test_dir = os.path.join(output_dir, 'test')
val_dir = os.path.join(output_dir, 'val')

In [4]:
# Create output directories if they don't exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

In [5]:
# Function to load and preprocess images from a folder
def load_images_from_folder(folder_path, label):
    images = []
    labels = []
    for filename in os.listdir(folder_path):
        img = cv2.imread(os.path.join(folder_path, filename))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB format
        img = cv2.resize(img, (512, 512))  # Resize to 512x512
        images.append(img)
        labels.append(label)
    return images, labels

In [6]:
# Load images from each class folder
healthy_images, healthy_labels = load_images_from_folder(healthy_path, label=0)
caterpillar_images, caterpillar_labels = load_images_from_folder(caterpillar_path, label=1)
leaf_spot_images, leaf_spot_labels = load_images_from_folder(leaf_spot_path, label=2)

In [7]:
# Split the data into train and test sets (80% train, 20% test)
healthy_train_images, healthy_test_images, healthy_train_labels, healthy_test_labels = train_test_split(healthy_images, healthy_labels, test_size=0.2, random_state=13)
caterpillar_train_images, caterpillar_test_images, caterpillar_train_labels, caterpillar_test_labels = train_test_split(caterpillar_images, caterpillar_labels, test_size=0.2, random_state=13)
leaf_spot_train_images, leaf_spot_test_images, leaf_spot_train_labels, leaf_spot_test_labels = train_test_split(leaf_spot_images, leaf_spot_labels, test_size=0.2, random_state=13)

In [8]:
# Print counts of images in each training and test set
print("Healthy Training Images:", len(healthy_train_images))
print("Healthy Test Images:", len(healthy_test_images))

print("Caterpillar Training Images:", len(caterpillar_train_images))
print("Caterpillar Test Images:", len(caterpillar_test_images))

print("Leaf Spot Training Images:", len(leaf_spot_train_images))
print("Leaf Spot Test Images:", len(leaf_spot_test_images))

print("Total Training Images", (len(healthy_train_images)+len(caterpillar_train_images)+len(leaf_spot_train_images)))
print("Total Testing Images", (len(healthy_test_images)+len(caterpillar_test_images)+len(leaf_spot_test_images)))

Healthy Training Images: 108
Healthy Test Images: 28
Caterpillar Training Images: 132
Caterpillar Test Images: 33
Leaf Spot Training Images: 94
Leaf Spot Test Images: 24
Total Training Images 334
Total Testing Images 85


In [9]:
# Further split the train data into train and validation sets (90% train, 20% validation)
healthy_train_images, healthy_val_images, healthy_train_labels, healthy_val_labels = train_test_split(healthy_train_images, healthy_train_labels, test_size=0.2, random_state=13)
caterpillar_train_images, caterpillar_val_images, caterpillar_train_labels, caterpillar_val_labels = train_test_split(caterpillar_train_images, caterpillar_train_labels, test_size=0.2, random_state=13)
leaf_spot_train_images, leaf_spot_val_images, leaf_spot_train_labels, leaf_spot_val_labels = train_test_split(leaf_spot_train_images, leaf_spot_train_labels, test_size=0.2, random_state=13)

In [10]:
# Print counts of images in each training and test set
print("Healthy Training Images:", len(healthy_train_images))
print("Healthy val Images:", len(healthy_val_images))

print("Caterpillar Training Images:", len(caterpillar_train_images))
print("Caterpillar val Images:", len(caterpillar_val_images))

print("Leaf Spot Training Images:", len(leaf_spot_train_images))
print("Leaf Spot val Images:", len(leaf_spot_val_images))

print("Total Training Images", (len(healthy_train_images)+len(caterpillar_train_images)+len(leaf_spot_train_images)))
print("Total validation Images", (len(healthy_val_images)+len(caterpillar_val_images)+len(leaf_spot_val_images)))

Healthy Training Images: 86
Healthy val Images: 22
Caterpillar Training Images: 105
Caterpillar val Images: 27
Leaf Spot Training Images: 75
Leaf Spot val Images: 19
Total Training Images 266
Total validation Images 68


In [11]:
# Combine the train images and labels from all classes
train_images = np.concatenate((healthy_train_images, caterpillar_train_images, leaf_spot_train_images), axis=0)
train_labels = np.concatenate((healthy_train_labels, caterpillar_train_labels, leaf_spot_train_labels), axis=0)

In [12]:
# Create data generator
datagen = ImageDataGenerator(
    rotation_range=45,  # Rotate by +45 and -45 degrees
    width_shift_range=0.1,  # Apply random horizontal shift
    height_shift_range=0.1,  # Apply random vertical shift
    horizontal_flip=True,  # Apply horizontal flip
    shear_range=0.0,  # No shear transformations
    zoom_range=0.0,  # No zoom transformations
    brightness_range=None  # No brightness adjustments
)

In [13]:
# Augment the training data
augmented_images = []
augmented_labels = []
for img, label in zip(train_images, train_labels):
    for _ in range(9):  # Generate 9 augmented images for each original image
        augmented_img = datagen.random_transform(img)
        augmented_images.append(augmented_img)
        augmented_labels.append(label)

In [14]:
# Combine original and augmented data
augmented_train_images = np.concatenate((train_images, augmented_images), axis=0)
augmented_train_labels = np.concatenate((train_labels, augmented_labels), axis=0)

In [15]:
# Shuffle the training data
shuffled_indices = np.arange(len(augmented_train_images))
np.random.shuffle(shuffled_indices)
augmented_train_images = augmented_train_images[shuffled_indices]
augmented_train_labels = augmented_train_labels[shuffled_indices]

In [16]:
# Print the count of images in the train set
print("Count of images in the train set:", len(augmented_train_images))

Count of images in the train set: 2660


In [17]:
# Define a function to save images to a folder
def save_images_to_folder(images, labels, folder_path):
    for img, label in zip(images, labels):
        class_folder = os.path.join(folder_path, str(label))
        os.makedirs(class_folder, exist_ok=True)
        # Generate a unique filename for each image
        filename = os.path.join(class_folder, f"image_{random.randint(1, 1000000)}.jpg")
        cv2.imwrite(filename, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

In [18]:
# Save augmented training images
save_images_to_folder(augmented_train_images, augmented_train_labels, train_dir)

In [ ]:
# Save testing images
save_images_to_folder(healthy_test_images, healthy_test_labels, test_dir)
save_images_to_folder(caterpillar_test_images, caterpillar_test_labels, test_dir)
save_images_to_folder(leaf_spot_test_images, leaf_spot_test_labels, test_dir)

In [ ]:
# Save validation images
save_images_to_folder(healthy_val_images, healthy_val_labels, val_dir)
save_images_to_folder(caterpillar_val_images, caterpillar_val_labels, val_dir)
save_images_to_folder(leaf_spot_val_images, leaf_spot_val_labels, val_dir)